## Task Definition: Instruction-Based Academic Paper Summarization

We study supervised fine-tuning of pretrained encoder–decoder language models
for academic paper summarization.

**Input**:  
A full academic paper (or long-form article text), provided together with an
explicit natural-language instruction describing how the summary should be written.

**Output**:  
A concise, abstract-style summary that:
- preserves key technical terminology,
- emphasizes the main contributions and results,
- uses formal academic tone,
- omits low-level implementation details.

The task is framed as instruction-following summarization, enabling controlled
experiments on model architecture, prompt structure, and parameter-efficient
adaptation capacity.


## Install RapidFire AI Package and Services


In [ ]:
!pip -q install bitsandbytes
!pip -q uninstall -y transformers trl
!pip -q install "transformers==4.56.1" "trl==0.21.0"


In [ ]:
import importlib.util, sys
print("bnb installed?", importlib.util.find_spec("bitsandbytes") is not None)
print("python:", sys.version)


In [ ]:
try:
    import rapidfireai
    print("✅ rapidfireai already installed")
except ImportError:
    %pip install rapidfireai  # Takes 1 min
    !rapidfireai init # Takes 1 min

In [ ]:
import transformers, trl
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)


## Start RapidFire Services


In [ ]:
import subprocess
from time import sleep
import socket
try:
  s = [socket.socket(socket.AF_INET, socket.SOCK_STREAM), socket.socket(socket.AF_INET, socket.SOCK_STREAM), socket.socket(socket.AF_INET, socket.SOCK_STREAM)]
  s[0].connect(("127.0.0.1", 8851))
  s[1].connect(("127.0.0.1", 8852))
  s[2].connect(("127.0.0.1", 8853))
  s[0].close()
  s[1].close()
  s[2].close()
  print("RapidFire Services are running")
except OSError as error:
  print("RapidFire Services are not running, launching now...")
  subprocess.Popen(["rapidfireai", "start"])
  sleep(30)

## Configure RapidFire to Use TensorBoard


In [ ]:
import os

# Load TensorBoard extension
%load_ext tensorboard

# TensorBoard log directory will be auto-created in experiment path

## Import RapidFire Components


In [ ]:
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

# NB: If you get "AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'" from Colab, just rerun this cell

## Dataset

We use the **arXiv academic summarization dataset**, which provides pairs of:

- **Article**: full paper text
- **Abstract**: human-written gold-standard abstract

The dataset contains approximately:
- 203k training examples
- 6k validation examples
- 6k test examples

This dataset directly aligns with the academic summarization task and enables
objective evaluation using reference abstracts.


## Base Model Architectures

We compare two pretrained encoder–decoder transformer architectures:

**T5 (Text-to-Text Transfer Transformer)**  
T5 is pretrained in a task-agnostic text-to-text framework, which provides a
strong inductive bias toward instruction following and structured generation.

**BART (Bidirectional and Auto-Regressive Transformer)**  
BART is pretrained using a denoising objective, encouraging reconstruction and
compression, which may be advantageous for abstractive summarization.

By comparing these models under identical fine-tuning conditions, we study how
pretraining inductive biases affect academic summarization performance.


In [ ]:
# !pip -q uninstall -y pyarrow datasets
# !pip -q install -U "pyarrow<23" "datasets>=3.0.0,<5"


In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset("ccdv/arxiv-summarization")

# Inspect columns (optional, for sanity)
print(dataset)
print(dataset["train"].column_names)

# REDUCED dataset for Colab stability (IMPORTANT)
train_dataset = dataset["train"].select(range(64)).shuffle(seed=42)      # 64 examples
eval_dataset  = dataset["validation"].select(range(16)).shuffle(seed=42) # 16 examples

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))



In [ ]:
T5_MODEL_NAME = "google/flan-t5-small"
BART_MODEL_NAME = "facebook/bart-base"


## Input Truncation Policy

Academic papers often exceed the maximum context length of encoder–decoder models.
We therefore truncate the paper text to the first 2048 tokens.

This choice reflects the empirical observation that introductions and contribution
statements—most relevant for abstract writing—occur early in the document.


### Lock the baseline instruction so all models start from the same task definition

In [ ]:
BASE_SUMMARIZATION_INSTRUCTION = (
    "Summarize the following academic paper into a concise abstract. "
    "Preserve key technical terms, emphasize the main contributions and results, "
    "and use formal academic tone."
)


## Define Data Processing Function


**This function’s only job is to package each training example into the exact (input_text → target_text) format that the base models expect during supervised fine-tuning.**

In [ ]:
def format_example(ex):
    return {
        "input_text": f"{BASE_SUMMARIZATION_INSTRUCTION}\n\nPaper:\n{ex['article']}",
        "target_text": ex["abstract"],
    }

# Apply mapping (creates the exact columns we train on)
train_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
eval_dataset  = eval_dataset.map(format_example,  remove_columns=eval_dataset.column_names)

print(train_dataset[0].keys())
print(train_dataset[0]["input_text"][:300])
print(train_dataset[0]["target_text"][:300])



In [ ]:
from transformers import AutoTokenizer

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_source_len = 512   # T4-safe
max_target_len = 128   # T4-safe

def tokenize_seq2seq_fixed(batch):
    # Encoder: fixed-length padding
    enc = tokenizer(
        batch["input_text"],
        max_length=max_source_len,
        truncation=True,
        padding="max_length",
    )

    # Decoder targets: fixed-length padding
    dec = tokenizer(
        batch["target_text"],
        max_length=max_target_len,
        truncation=True,
        padding="max_length",
    )

    labels = dec["input_ids"]
    pad_id = tokenizer.pad_token_id

    # Convert pad tokens to -100 so loss ignores them
    labels = [[(-100 if tok == pad_id else tok) for tok in seq] for seq in labels]
    enc["labels"] = labels
    return enc

tokenized_train_T5 = train_dataset.map(
    tokenize_seq2seq_fixed,
    batched=True,
    remove_columns=train_dataset.column_names,
)
tokenized_eval_T5 = eval_dataset.map(
    tokenize_seq2seq_fixed,
    batched=True,
    remove_columns=eval_dataset.column_names,
)

print(tokenized_train_T5.column_names)  # expect: input_ids, attention_mask, labels


### BART Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name_BART = "facebook/bart-base"
tokenizer_BART = AutoTokenizer.from_pretrained(model_name_BART)

max_source_len = 512   # T4-safe
max_target_len = 128   # T4-safe

In [ ]:
def tokenize_seq2seq_fixed_BART(batch):
    # Encoder: fixed-length padding
    enc = tokenizer_BART(
        batch["input_text"],
        max_length=max_source_len,
        truncation=True,
        padding="max_length",
    )

    # Decoder targets: fixed-length padding
    dec = tokenizer_BART(
        batch["target_text"],
        max_length=max_target_len,
        truncation=True,
        padding="max_length",
    )

    labels = dec["input_ids"]
    pad_id = tokenizer_BART.pad_token_id

    # Convert pad tokens to -100 so loss ignores them
    labels = [[(-100 if tok == pad_id else tok) for tok in seq] for seq in labels]
    enc["labels"] = labels
    return enc

In [ ]:
tokenized_train_BART = train_dataset.map(
    tokenize_seq2seq_fixed_BART,
    batched=True,
    remove_columns=train_dataset.column_names,
)
tokenized_eval_BART = eval_dataset.map(
    tokenize_seq2seq_fixed_BART,
    batched=True,
    remove_columns=eval_dataset.column_names,
)

print(tokenized_train_BART.column_names)  # expect: input_ids, attention_mask, labels

In [ ]:
import numpy as np

def count_all_masked(ds, n=200):
    n = min(n, len(ds))
    all_masked = 0
    min_valid = 10**9
    max_valid = 0

    for i in range(n):
        labs = np.array(ds[i]["labels"])
        valid = int((labs != -100).sum())

        if valid == 0:
            all_masked += 1

        min_valid = min(min_valid, valid)
        max_valid = max(max_valid, valid)

    print("Checked:", n)
    print("All-masked examples:", all_masked)
    print("Min valid label tokens:", min_valid)
    print("Max valid label tokens:", max_valid)

count_all_masked(tokenized_train_T5, n=200)
count_all_masked(tokenized_eval_T5, n=200)

In [ ]:
ex = tokenized_train_T5[0]
print(len(ex["input_ids"]), len(ex["labels"]))
print(ex["labels"][:20])


## Define Metrics Function


In [ ]:

def sample_compute_metrics_T5(eval_preds):
    import numpy as np
    import evaluate
    from transformers import AutoTokenizer

    model_name = "google/flan-t5-small"  # worker-safe (no notebook globals)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    rouge = evaluate.load('rouge')


    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [s.strip() for s in decoded_preds]
    decoded_labels = [s.strip() for s in decoded_labels]

    scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
        rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"],
    )
    return {k: float(v) for k, v in scores.items()}



In [ ]:
!pip -q install loguru

## Initialize Experiment


In [ ]:
# EXPERIMENT_NAME = "academic_summarization_baseline"
# experiment = Experiment(experiment_name=EXPERIMENT_NAME)

## Get TensorBoard Log Directory


In [ ]:
# # Get experiment path
# from rapidfireai.fit.db.rf_db import RfDb

# db = RfDb()
# experiment_path = db.get_experiments_path(EXPERIMENT_NAME)
# tensorboard_log_dir = f"{experiment_path}/tensorboard_logs/{EXPERIMENT_NAME}"

# print(f"TensorBoard logs will be saved to: {tensorboard_log_dir}")

## Define Model Configurations


In [ ]:
# from peft import TaskType
# peft_configs = List([
#     RFLoraConfig(
#  r=16,
#  lora_alpha=32,
#  target_modules=["q", "v"],
#  lora_dropout=0.05,
#  bias="none",
#  task_type=TaskType.SEQ_2_SEQ_LM
#   )

# ])

# config_set = List([
#     RFModelConfig(
#         model_name=T5_MODEL_NAME,
#         model_type="seq2seq_lm",
#         model_kwargs={
#             "device_map": "auto",
#             "torch_dtype": "float16",
#             "use_cache": False,
#         },
#         peft_config=peft_configs,
#         compute_metrics=sample_compute_metrics_T5,
#         training_args=RFSFTConfig(
#             # short + guaranteed logging (so TensorBoard will show scalars)
#             max_steps=30,
#             logging_steps=1,
#             eval_strategy="steps",
#             eval_steps=5,

#             learning_rate=2e-4,
#             lr_scheduler_type="linear",
#             per_device_train_batch_size=1,
#             per_device_eval_batch_size=1,
#             gradient_accumulation_steps=1,  # keep small while debugging
#             fp16=True,
#             gradient_checkpointing=True,

#             report_to="tensorboard",  # IMPORTANT for scalars
#         ),
#     )
# ])
# print("✅ config_group ready")

In [ ]:
# def sample_create_model(model_config):
#     """Create model + tokenizer (sample notebook style), adapted for seq2seq."""
#     from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
#     import torch

#     model_name  = model_config["model_name"]
#     model_kwargs = model_config.get("model_kwargs", {}).copy()

#     if model_kwargs.get("torch_dtype") == "float16":
#         model_kwargs["torch_dtype"] = torch.float16
#     elif model_kwargs.get("torch_dtype") == "bfloat16":
#         model_kwargs["torch_dtype"] = torch.bfloat16

#     model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
#     tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

#     model.config.use_cache = False
#     return (model, tokenizer)

In [ ]:
# config_group = RFGridSearch(
#     configs=config_set,
#     trainer_type="SFT"
# )

## Start TensorBoard


In [ ]:
# %tensorboard --logdir {tensorboard_log_dir}

In [ ]:
# import transformers, trl
# print("transformers:", transformers.__version__)
# print("trl:", trl.__version__)

## Run Training + Validation


In [ ]:
# experiment.run_fit(
#     config_group,
#     sample_create_model,
#     tokenized_train_T5,
#     tokenized_eval_T5,
#     num_chunks=1,  # keep 1 for Colab stability
#     seed=42,
# )

The training process initiated by `run_fit` runs asynchronously. It can take some time for the experiment to start, load the model, and process enough steps to generate its first logs. The `InteractiveController` relies on these logs to show progress. Let's try waiting a bit longer and then display the controller again.

## Launch Interactive Run Controller


In [ ]:
# from time import sleep
# from rapidfireai.fit.utils.interactive_controller import InteractiveController

# # Wait for a longer period to allow the experiment to initialize and log progress
# print("Waiting 60 seconds for the experiment to start and generate logs...")
# sleep(60)

# # Re-attempt to display the interactive controller
# controller = InteractiveController(dispatcher_url="http://127.0.0.1:8851")
# controller.display()
# print("If still no runs are found, try refreshing the page or checking the TensorBoard logs as described in the RapidFire AI status output.")

## End Experiment


In [ ]:
# from google.colab import output
# from IPython.display import display, HTML

# display(HTML('''
# <button id="continue-btn" style="padding: 10px 20px; font-size: 16px;">Click to End Experiment</button>
# '''))

# # eval_js blocks until the Promise resolves
# output.eval_js('''
# new Promise((resolve) => {
#     document.getElementById("continue-btn").onclick = () => {
#         document.getElementById("continue-btn").disabled = true;
#         document.getElementById("continue-btn").innerText = "Continuing...";
#         resolve("clicked");
#     };
# })
# ''')

# # Actually end the experiment after the button is clicked
# experiment.end()
# print("Done!")

In [ ]:
#

## View TensorBoard Plots and Logs


In [ ]:
# # View final logs
# %tensorboard --logdir {tensorboard_log_dir}

## View RapidFire AI Log Files


In [ ]:
# # Get the experiment-specific log file
# from IPython.display import display, Pretty
# log_file = experiment.get_log_file_path()

# display(Pretty(f"📄 Experiment Log File: {log_file}"))

# if log_file.exists():
#     display(Pretty("=" * 80))
#     display(Pretty(f"Last 30 lines of {log_file.name}:"))
#     display(Pretty("=" * 80))
#     with open(log_file, 'r', encoding='utf-8') as f:
#         lines = f.readlines()
#         for line in lines[-30:]:
#             display(Pretty(line.rstrip()))
# else:
#     display(Pretty(f"❌ Log file not found: {log_file}"))

In [ ]:
# # Get the training-specific log file
# log_file = experiment.get_log_file_path("training")

# display(Pretty(f"📄 Training Log File: {log_file}"))

# if log_file.exists():
#     display(Pretty("=" * 80))
#     display(Pretty(f"Last 30 lines of {log_file.name}:"))
#     display(Pretty("=" * 80))
#     with open(log_file, 'r', encoding='utf-8') as f:
#         lines = f.readlines()
#         for line in lines[-30:]:
#             display(Pretty(line.rstrip()))
# else:
#     display(Pretty(f"❌ Log file not found: {log_file}"))

# **EXPERIMENT A: LoRA comparisons**

### Experiment A — LoRA Rank Ablation
We compare LoRA rank r=8 vs r=16 while holding the base model, prompt scheme, and training budget constant. This isolates the effect of adapter capacity on summarization quality (ROUGE, eval loss).


In [ ]:
# EXPERIMENT_NAME = "academic_summarization_expA_lora_rank"
# experiment = Experiment(experiment_name=EXPERIMENT_NAME)
# print("✅ Initialized:", EXPERIMENT_NAME)

In [ ]:
# from rapidfireai.fit.db.rf_db import RfDb

# db = RfDb()
# experiment_path = db.get_experiments_path(EXPERIMENT_NAME)
# tensorboard_log_dir = f"{experiment_path}/tensorboard_logs/{EXPERIMENT_NAME}"

# print(f"TensorBoard logs will be saved to: {tensorboard_log_dir}")

In [ ]:
# from peft import TaskType

# peft_configs = List([
#     RFLoraConfig(
#         r=8,
#         lora_alpha=32,
#         target_modules=["q", "v"],
#         lora_dropout=0.05,
#         bias="none",
#         task_type=TaskType.SEQ_2_SEQ_LM
#     ),
#     RFLoraConfig(
#         r=16,
#         lora_alpha=32,
#         target_modules=["q", "v"],
#         lora_dropout=0.05,
#         bias="none",
#         task_type=TaskType.SEQ_2_SEQ_LM
#     ),
# ])

# config_set = List([
#     RFModelConfig(
#         model_name=T5_MODEL_NAME,
#         model_type="seq2seq_lm",
#         model_kwargs={
#             "device_map": "auto",
#             "torch_dtype": "float16",
#             "use_cache": False,
#         },
#         peft_config=peft_configs,
#         compute_metrics=sample_compute_metrics_T5,
#         training_args=RFSFTConfig(
#             max_steps=30,
#             logging_steps=1,
#             eval_strategy="steps",
#             eval_steps=5,
#             learning_rate=2e-4,
#             lr_scheduler_type="linear",
#             per_device_train_batch_size=1,
#             per_device_eval_batch_size=1,
#             gradient_accumulation_steps=1,
#             fp16=True,
#             gradient_checkpointing=True,
#             report_to="tensorboard",
#         ),
#     )
# ])

# config_group = RFGridSearch(configs=config_set, trainer_type="SFT")
# print("✅ Experiment A config_group ready")

In [ ]:
# %tensorboard --logdir {tensorboard_log_dir}

In [ ]:
# experiment.run_fit(
#     config_group,
#     sample_create_model,
#     tokenized_train_T5,
#     tokenized_eval_T5,
#     num_chunks=1,
#     seed=42,
# )

In [ ]:
# from google.colab import output
# from IPython.display import display, HTML

# display(HTML('''
# <button id="continue-btn" style="padding: 10px 20px; font-size: 16px;">Click to End Experiment</button>
# '''))

# # eval_js blocks until the Promise resolves
# output.eval_js('''
# new Promise((resolve) => {
#     document.getElementById("continue-btn").onclick = () => {
#         document.getElementById("continue-btn").disabled = true;
#         document.getElementById("continue-btn").innerText = "Continuing...";
#         resolve("clicked");
#     };
# })
# ''')

# # Actually end the experiment after the button is clicked
# experiment.end()
# print("Done!")

In [ ]:
# # Get the experiment-specific log file
# from IPython.display import display, Pretty
# log_file = experiment.get_log_file_path()

# display(Pretty(f"📄 Experiment Log File: {log_file}"))

# if log_file.exists():
#     display(Pretty("=" * 80))
#     display(Pretty(f"Last 30 lines of {log_file.name}:"))
#     display(Pretty("=" * 80))
#     with open(log_file, 'r', encoding='utf-8') as f:
#         lines = f.readlines()
#         for line in lines[-30:]:
#             display(Pretty(line.rstrip()))
# else:
#     display(Pretty(f"❌ Log file not found: {log_file}"))

In [ ]:
# # Get the training-specific log file
# log_file = experiment.get_log_file_path("training")

# display(Pretty(f"📄 Training Log File: {log_file}"))

# if log_file.exists():
#     display(Pretty("=" * 80))
#     display(Pretty(f"Last 30 lines of {log_file.name}:"))
#     display(Pretty("=" * 80))
#     with open(log_file, 'r', encoding='utf-8') as f:
#         lines = f.readlines()
#         for line in lines[-30:]:
#             display(Pretty(line.rstrip()))
# else:
#     display(Pretty(f"❌ Log file not found: {log_file}"))

In [ ]:
# View final logs
# %tensorboard --logdir {tensorboard_log_dir}

# Experiment B: Base Model Architecture

## Experiment B: T5

In [ ]:
### ExpB_T5: Experiment Initialization

# EXPERIMENT_NAME = "academic_summarization_expB__T5_base"
# experiment = Experiment(experiment_name=EXPERIMENT_NAME)
# print("✅ Initialized:", EXPERIMENT_NAME)

In [ ]:
# # Get experiment path
# from rapidfireai.fit.db.rf_db import RfDb

# db = RfDb()
# experiment_path = db.get_experiments_path(EXPERIMENT_NAME)
# tensorboard_log_dir = f"{experiment_path}/tensorboard_logs/{EXPERIMENT_NAME}"

# print(f"TensorBoard logs will be saved to: {tensorboard_log_dir}")

In [ ]:
### ExpB_T5: Model Factory

# def sample_create_model_T5(model_config):
#     """Create model + tokenizer (sample notebook style), adapted for seq2seq."""
#     from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
#     import torch

#     model_name = model_config["model_name"]
#     model_kwargs = model_config.get("model_kwargs", {}).copy()

#     # Convert torch_dtype string to torch dtype if needed
#     if "torch_dtype" in model_kwargs and isinstance(model_kwargs["torch_dtype"], str):
#         if model_kwargs["torch_dtype"] == "float16":
#             model_kwargs["torch_dtype"] = torch.float16

#     model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
#     tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

#     model.config.use_cache = False

#     return (model, tokenizer)

In [ ]:
## ExpB_T5: RapidFire Configuration

from peft import TaskType

peft_configs_T5_ExpB = List([
    RFLoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q", "v"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM
    )
])

config_set_T5_ExpB = List([
    RFModelConfig(
        model_name="google/flan-t5-small",
        model_type="seq2seq_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "float16",
            "use_cache": False,
        },
        peft_config=peft_configs_T5_ExpB,
        compute_metrics=sample_compute_metrics_T5,
        training_args=RFSFTConfig(
            max_steps=30,
            max_grad_norm = 1.0,
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=5,
            learning_rate=5e-5, #changed from 2e-4
            lr_scheduler_type="linear",
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=1,
            fp16=False, ##changed coz of 0-loss
            gradient_checkpointing=True,
            report_to="tensorboard",
        ),
    )
])

config_group_T5_ExpB = RFGridSearch(
    configs=config_set_T5_ExpB,
    trainer_type="SFT"
)

print("✅ ExpB_T5 config_group ready")

In [ ]:
# %tensorboard --logdir {tensorboard_log_dir}

In [ ]:
### ExpB_T5: Run Training

# experiment.run_fit(
#     config_group_T5_ExpB,
#     sample_create_model_T5,
#     tokenized_train_T5,
#     tokenized_eval_T5,
#     num_chunks=1,
#     seed=42,
# )

In [ ]:
# # Create Interactive Controller
# sleep(15)
# from rapidfireai.fit.utils.interactive_controller import InteractiveController

# controller = InteractiveController(dispatcher_url="http://127.0.0.1:8851")
# controller.display()

In [ ]:
# from google.colab import output
# from IPython.display import display, HTML

# display(HTML('''
# <button id="continue-btn" style="padding: 10px 20px; font-size: 16px;">Click to End Experiment</button>
# '''))

# # eval_js blocks until the Promise resolves
# output.eval_js('''
# new Promise((resolve) => {
#     document.getElementById("continue-btn").onclick = () => {
#         document.getElementById("continue-btn").disabled = true;
#         document.getElementById("continue-btn").innerText = "Continuing...";
#         resolve("clicked");
#     };
# })
# ''')

# # Actually end the experiment after the button is clicked
# experiment.end()
# print("Done!")

In [ ]:
# %tensorboard --logdir {tensorboard_log_dir}

In [ ]:
# # Get the experiment-specific log file
# from IPython.display import display, Pretty
# log_file = experiment.get_log_file_path()

# display(Pretty(f"📄 Experiment Log File: {log_file}"))

# if log_file.exists():
#     display(Pretty("=" * 80))
#     display(Pretty(f"Last 30 lines of {log_file.name}:"))
#     display(Pretty("=" * 80))
#     with open(log_file, 'r', encoding='utf-8') as f:
#         lines = f.readlines()
#         for line in lines[-30:]:
#             display(Pretty(line.rstrip()))
# else:
#     display(Pretty(f"❌ Log file not found: {log_file}"))

In [ ]:
# # Get the training-specific log file
# log_file = experiment.get_log_file_path("training")

# display(Pretty(f"📄 Training Log File: {log_file}"))

# if log_file.exists():
#     display(Pretty("=" * 80))
#     display(Pretty(f"Last 30 lines of {log_file.name}:"))
#     display(Pretty("=" * 80))
#     with open(log_file, 'r', encoding='utf-8') as f:
#         lines = f.readlines()
#         for line in lines[-30:]:
#             display(Pretty(line.rstrip()))
# else:
#     display(Pretty(f"❌ Log file not found: {log_file}"))

## Experiment B: BART

In [ ]:
### ExpB_BART: Experiment Initialization

EXPERIMENT_NAME = "academic_summarization_expB__BART_base"
experiment = Experiment(experiment_name=EXPERIMENT_NAME)
print("✅ Initialized:", EXPERIMENT_NAME)

In [ ]:
# Get experiment path
from rapidfireai.fit.db.rf_db import RfDb

db = RfDb()
experiment_path = db.get_experiments_path(EXPERIMENT_NAME)
tensorboard_log_dir = f"{experiment_path}/tensorboard_logs/{EXPERIMENT_NAME}"

print(f"TensorBoard logs will be saved to: {tensorboard_log_dir}")

In [ ]:
def sample_compute_metrics_BART(eval_preds):
    import numpy as np
    import evaluate
    from transformers import AutoTokenizer

    model_name = "facebook/bart-base"  # worker-safe (no notebook globals)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    rouge = evaluate.load('rouge')


    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    if hasattr(preds, "ndim") and preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [s.strip() for s in decoded_preds]
    decoded_labels = [s.strip() for s in decoded_labels]

    scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
        rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"],
    )
    return {k: float(v) for k, v in scores.items()}

In [ ]:
### ExpB_BART: RapidFire Configuration

from peft import TaskType

peft_config_BART_ExpB = RFLoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

config_set_BART_ExpB = [
    RFModelConfig(
        model_name="facebook/bart-base",
        model_type="seq2seq_lm",
        model_kwargs={
            "device_map": "auto",
            "torch_dtype": "float32",
            "use_cache": False,
        },
        peft_config=peft_config_BART_ExpB,   # <-- SINGLE object
        compute_metrics=sample_compute_metrics_BART,
        training_args=RFSFTConfig(
            max_steps=30,
            max_grad_norm=1.0,
            logging_steps=1,
            eval_strategy="steps",
            eval_steps=5,
            learning_rate=5e-5,
            lr_scheduler_type="linear",
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=1,
            fp16=False,
            gradient_checkpointing=True,
            report_to=["tensorboard"],
        ),
    )
]


In [ ]:
### ExpB_BART: Model Factory
### ExpB_BART: Model Factory (LoRA applied here)

def sample_create_model_BART(model_config):
    """Create BART model + tokenizer and apply LoRA inside factory (debuggable)."""
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    import torch
    from peft import get_peft_model

    model_name = model_config["model_name"]
    model_kwargs = model_config.get("model_kwargs", {}).copy()

    if "torch_dtype" in model_kwargs and isinstance(model_kwargs["torch_dtype"], str):
        if model_kwargs["torch_dtype"] == "float16":
            model_kwargs["torch_dtype"] = torch.float16
        elif model_kwargs["torch_dtype"] == "float32":
            model_kwargs["torch_dtype"] = torch.float32

    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)


    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.train()

    # ---- Apply LoRA HERE so we can verify trainables ----
    peft_cfg = model_config.get("peft_config", None)
    if peft_cfg is not None:
        model = get_peft_model(model, peft_cfg)

    # ✅ HARD CHECK (this is the check that prevents your crash)
    trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    print("✅ Trainable tensors:", len(trainable))
    print("✅ Example trainables:", [n for n, _ in trainable[:25]])

    assert len(trainable) > 0, (
        "❌ No trainable params. LoRA target_modules did not match BART modules."
    )

    return (model, tokenizer)

In [ ]:
config_group_BART_ExpB = RFGridSearch(
    configs=config_set_BART_ExpB,
    trainer_type="SFT"
)

print("✅ ExpB_BART config_group ready")

In [ ]:
%tensorboard --logdir {tensorboard_log_dir}

In [ ]:
# Verify model has trainable parameters before running experiment
print("=" * 80)
print("Checking trainable parameters for BART model...")
print("=" * 80)

# Handle both list and single config object
test_config = config_set_BART_ExpB[0] if isinstance(config_set_BART_ExpB, (list, tuple)) else config_set_BART_ExpB
model, tokenizer = sample_create_model_BART(test_config)

# Show trainable parameters summary
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
non_trainable_params = total_params - trainable_params

print(f"\n📊 Parameter Summary:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Non-trainable parameters: {non_trainable_params:,}")
print(f"   Trainable percentage: {100 * trainable_params / total_params:.2f}%")

if trainable_params == 0:
    print("\n⚠️  WARNING: No trainable parameters found! This will cause gradient errors.")
else:
    print(f"\n✅ Model has {trainable_params:,} trainable parameters - ready for training")
    
print("=" * 80)

In [ ]:
### ExpB_BART: Run Training

experiment.run_fit(
    config_group_BART_ExpB,
    sample_create_model_BART,
    tokenized_train_BART,
    tokenized_eval_BART,
    num_chunks=1,
    seed=42,
)

In [ ]:
# Create Interactive Controller
sleep(15)
from rapidfireai.fit.utils.interactive_controller import InteractiveController

controller = InteractiveController(dispatcher_url="http://127.0.0.1:8851")
controller.display()

In [ ]:
from google.colab import output
from IPython.display import display, HTML

display(HTML('''
<button id="continue-btn" style="padding: 10px 20px; font-size: 16px;">Click to End Experiment</button>
'''))

# eval_js blocks until the Promise resolves
output.eval_js('''
new Promise((resolve) => {
    document.getElementById("continue-btn").onclick = () => {
        document.getElementById("continue-btn").disabled = true;
        document.getElementById("continue-btn").innerText = "Continuing...";
        resolve("clicked");
    };
})
''')

# Actually end the experiment after the button is clicked
experiment.end()
print("Done!")

In [ ]:
%tensorboard --logdir {tensorboard_log_dir}

In [ ]:
# Get the experiment-specific log file
from IPython.display import display, Pretty
log_file = experiment.get_log_file_path()

display(Pretty(f"📄 Experiment Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))

In [ ]:
# Get the training-specific log file
log_file = experiment.get_log_file_path("training")

display(Pretty(f"📄 Training Log File: {log_file}"))

if log_file.exists():
    display(Pretty("=" * 80))
    display(Pretty(f"Last 30 lines of {log_file.name}:"))
    display(Pretty("=" * 80))
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            display(Pretty(line.rstrip()))
else:
    display(Pretty(f"❌ Log file not found: {log_file}"))